In [2]:
#imports
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

## Loading the Dataset

The training dataset contains features along with the target label.  
The test dataset contains only features.

The goal is to train on the training data and generate predictions for the test data.

In [3]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
train_df['Heart Disease'] = train_df['Heart Disease'].map({
    'Absence': 0,
    'Presence': 1
})
print(train_df.shape)
print(test_df.shape)
train_df.head()

(630000, 15)
(270000, 14)


,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,1
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,0
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,0
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,0
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,1


## Exploratory Data Analysis (EDA)

We check:
- Missing values
- Distribution of target variable

This helps us understand data quality and class balance.

In [4]:
train_df.isnull().sum()
train_df['Heart Disease'].value_counts(normalize=True)

Heart Disease
0    0.55166
1    0.44834
Name: proportion, dtype: float64

## Feature Engineering

New features are created to capture hidden relationships between variables.

Examples:
- Ratio of cholesterol to HDL
- Square of age

Feature engineering often improves model performance more than changing models.

In [5]:
def feature_engineering(df):
    df = df.copy()
    
    if 'cholesterol' in df.columns and 'hdl' in df.columns:
        df['chol_hdl_ratio'] = df['cholesterol'] / (df['hdl'] + 1)
        
    if 'age' in df.columns:
        df['age_squared'] = df['age'] ** 2
        
    return df

## Data Preparation

- Apply feature engineering
- Separate features and target
- Remove ID column

In [6]:
train_df = feature_engineering(train_df)
test_df = feature_engineering(test_df)

X = train_df.drop(columns=["id","Heart Disease"])
y = train_df["Heart Disease"]
X_test = test_df.drop(columns=["id"])

## Feature Scaling

Standardization is applied so that numerical features have similar ranges.

This helps gradient-based models converge faster.

In [7]:
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

## Stratified K-Fold Cross Validation

We use 5-fold Stratified CV to:
- Preserve class distribution
- Obtain reliable performance estimates

In [8]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## XGBoost Model

XGBoost is a gradient boosting algorithm that builds trees sequentially, where each tree corrects errors of previous trees.

In [9]:
xgb_preds = np.zeros(len(X_test))
xgb_oof = np.zeros(len(X))

for train_idx, val_idx in skf.split(X, y):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = XGBClassifier(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss"
    )

    model.fit(X_tr, y_tr)

    xgb_oof[val_idx] = model.predict_proba(X_val)[:,1]
    xgb_preds += model.predict_proba(X_test)[:,1] / 5

print("XGBoost LogLoss:", log_loss(y, xgb_oof))

XGBoost LogLoss: 0.2681789045226048


## LightGBM Model

LightGBM is optimized for speed and memory efficiency while maintaining high accuracy.

In [10]:
lgb_preds = np.zeros(len(X_test))
lgb_oof = np.zeros(len(X))

for train_idx, val_idx in skf.split(X, y):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8
    )

    model.fit(X_tr, y_tr)

    lgb_oof[val_idx] = model.predict_proba(X_val)[:,1]
    lgb_preds += model.predict_proba(X_test)[:,1] / 5

print("LightGBM LogLoss:", log_loss(y, lgb_oof))

[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013861 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 422
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012197 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wis

## CatBoost Model

CatBoost handles categorical features well and reduces overfitting through ordered boosting.

In [11]:
cat_preds = np.zeros(len(X_test))
cat_oof = np.zeros(len(X))

for train_idx, val_idx in skf.split(X, y):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        verbose=0
    )

    model.fit(X_tr, y_tr)

    cat_oof[val_idx] = model.predict_proba(X_val)[:,1]
    cat_preds += model.predict_proba(X_test)[:,1] / 5

print("CatBoost LogLoss:", log_loss(y, cat_oof))

CatBoost LogLoss: 0.2682205567862844


## Model Ensembling

Predictions from all three models are averaged to reduce variance and improve generalization.

In [12]:
final_preds = (xgb_preds + lgb_preds + cat_preds) / 3

## Create Submission File

The final predictions are saved in CSV format for Kaggle submission.

In [13]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "Heart Disease": final_preds
})

submission.to_csv("submission.csv", index=False)
submission.head()

,id,Heart Disease
0,630000,0.951369
1,630001,0.008478
2,630002,0.986057
3,630003,0.005693
4,630004,0.185961
